# BITS ID : 2026AJ05810
## BITS Pilani - M.Tech AIML - Machine Learning Assignment 2
### Wine Quality Binary Classification & Model Artifact Export

**Pipeline Steps:**
1. Download & preprocess UCI Red Wine Quality dataset (creating binary classification target `is_good_quality`).
2. Perform stratified 80/20 train/test split.
3. Save `test_data.csv` for evaluation & app testing.
4. Scale features using `StandardScaler` and persist preprocessor artifact.
5. Train and evaluate 5 mandatory classifiers across 6 standard performance metrics.
6. Display a formatted summary table and persist all trained models to the `model/` directory.

## Environment Setup & Imports

In [1]:
!pip install pandas numpy scikit-learn requests joblib

In [2]:
import os
import io
import requests
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# Ensuring artifact output directory exists
os.makedirs("model", exist_ok=True)
print("Environment initialized and 'model/' directory verified.")

Environment initialized and 'model/' directory verified.


## Data Ingestion & Target Definition

In [3]:
def load_wine_dataset(url: str = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv") -> pd.DataFrame:
    """Fetch UCI Red Wine dataset and transform into a binary classification problem."""
    print(f"[1/5] Downloading dataset from {url}...")
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    
    df = pd.read_csv(io.StringIO(response.text), sep=";")
    
    # Target definition: 1 if quality >= 7 (Good quality), else 0
    df["target"] = (df["quality"] >= 7).astype(int)
    df = df.drop(columns=["quality"])
    
    print(f"Dataset successfully loaded: {df.shape[0]} rows, {df.shape[1]} columns (11 features + 1 target).")
    return df

df = load_wine_dataset()
df.head()

[1/5] Downloading dataset from https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv...
Dataset successfully loaded: 1599 rows, 12 columns (11 features + 1 target).


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,target
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,0


## Data Splitting, Preprocessing & Test Export

In [4]:
# Separate features and target
X = df.drop(columns=["target"])
y = df["target"]

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Persist test set (features + target) for submission and app testing
test_df = X_test.copy()
test_df["target"] = y_test
test_df.to_csv("test_data.csv", index=False)
print(f"[2/5] Exported test dataset to 'test_data.csv' ({len(test_df)} samples).")

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler artifact
scaler_path = os.path.join("model", "preprocessor.joblib")
joblib.dump(scaler, scaler_path)
print(f"Scaler persisted to '{scaler_path}'.")

[2/5] Exported test dataset to 'test_data.csv' (320 samples).
Scaler persisted to 'model/preprocessor.joblib'.


## Model Definitions & Evaluation Loop

In [5]:
# 5 Mandated Classification Models
models = {
    "Logistic Regression": (LogisticRegression(random_state=42, max_iter=1000), "logistic_regression"),
    "Decision Tree": (DecisionTreeClassifier(random_state=42, max_depth=6), "decision_tree"),
    "KNN": (KNeighborsClassifier(n_neighbors=7), "knn"),
    "Naive Bayes": (GaussianNB(), "naive_bayes"),
    "Support Vector Machine": (SVC(probability=True, random_state=42, kernel="rbf"), "svm")
}

print("[3/5] Training models and computing metrics...")
results = []

for name, (model, file_prefix) in models.items():
    # Fit model
    model.fit(X_train_scaled, y_train)
    
    # Predict labels and probability estimates
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate performance metrics
    metrics = {
        "ML Model Name": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "AUC": round(roc_auc_score(y_test, y_proba), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "MCC": round(matthews_corrcoef(y_test, y_pred), 4)
    }
    results.append(metrics)
    
    # Persist model artifact
    save_path = os.path.join("model", f"{file_prefix}.joblib")
    joblib.dump(model, save_path)
    print(f"  -> Model '{name}' trained & saved to '{save_path}'.")

print("[4/5] All model artifacts saved inside 'model/' directory.")

[3/5] Training models and computing metrics...
  -> Model 'Logistic Regression' trained & saved to 'model/logistic_regression.joblib'.
  -> Model 'Decision Tree' trained & saved to 'model/decision_tree.joblib'.
  -> Model 'KNN' trained & saved to 'model/knn.joblib'.
  -> Model 'Naive Bayes' trained & saved to 'model/naive_bayes.joblib'.


/home/s2026aj05810/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


  -> Model 'Support Vector Machine' trained & saved to 'model/svm.joblib'.
[4/5] All model artifacts saved inside 'model/' directory.


## Summary Display

In [6]:
print("[5/5] Performance Summary Table:")
summary_df = pd.DataFrame(results)
summary_df.sort_values(by="AUC", ascending=False).reset_index(drop=True)

[5/5] Performance Summary Table:


,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Support Vector Machine,0.9000,0.8892,0.7619,0.3721,0.5000,0.4876
1,Logistic Regression,0.8938,0.8804,0.6957,0.3721,0.4848,0.4580
2,Decision Tree,0.9187,0.8611,0.7073,0.6744,0.6905,0.6440
3,Naive Bayes,0.8594,0.8517,0.4844,0.7209,0.5794,0.5131
4,KNN,0.8906,0.8215,0.6429,0.4186,0.5070,0.4617
